# AgentOps Lab 10 - When one agent becomes a team

The incident is now harder: checkout conversion has fallen 35% in Europe, but there is no obvious outage. The system must inspect logs, deployments, customer reports, metrics, runbooks, and historical incidents.

This notebook asks the article's key team-design question: what does this additional agent make meaningfully better than the simpler baseline?

## Specialist topology

```mermaid
flowchart TD
    C["Coordinator"] --> O["ObservabilityAgent"]
    C --> D["DeploymentAgent"]
    C --> U["CustomerImpactAgent"]
    O --> A["IncidentAnalystAgent"]
    D --> A
    U --> A
    A --> R["RiskReviewerAgent"]
    R --> REC["Recommendation"]
```

The goal is not more agents. The goal is better context management, clearer ownership, stronger evidence, and better risk review.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parents[1]
sys.path.insert(0, str(repo_root / "labs"))

from agentops_lab.multi_agent_team import compare_single_vs_team, run_multi_agent_team, run_single_agent_baseline


## Run the harder incident

The multi-agent team has higher cost and latency, but it improves accuracy by separating observability, deployment, customer impact, analysis, and risk review.

In [ ]:
comparison = compare_single_vs_team()
comparison["single_agent"], comparison["multi_agent_team"]


In [ ]:
for finding in comparison["findings"]:
    print(f"{finding['agent']}: {finding['finding']} confidence={finding['confidence']}")


## Compare trade-offs

| Metric | Single agent | Multi-agent team |
| --- | --- | --- |
| Accuracy | Lower on this hard incident | Higher because specialists cover missing context |
| Cost | Lower | Higher |
| Latency | Lower | Higher |
| Tool calls | Fewer | More |
| Coordination overhead | None | Real and measurable |

Often the single agent should win on simple incidents. That is intentional; teams are not a default.

In [ ]:
simple = compare_single_vs_team(simple_incident=True)
print("simple incident winner:", simple["decision"])
print(simple["single_agent"])


## Exercises

- Remove `RiskReviewerAgent`. What unsupported recommendation slips through?
- Merge `ObservabilityAgent` and `DeploymentAgent`. Does the team still justify itself?
- Add a max specialist budget and fail if coordination overhead exceeds the accuracy gain.
- Create an evaluation task where the single agent should win.

References: [Anthropic: Building a multi-agent research system](https://www.anthropic.com/engineering/multi-agent-research-system), [AutoGen paper](https://arxiv.org/abs/2308.08155), and [Building AI Agents: From Loops to Teams](https://www.linkedin.com/pulse/building-ai-agents-from-loops-teams-oneplusi-y3atc/).